# 01_syntgen: Synthetic patient generator
Run cells from top to bottom. After **Restart**, use **Run All**.

## Setup

In [ ]:
from datetime import date
import numpy as np, pandas as pd
import ipytest
ipytest.autoconfig()

CFG = {
    "seed": 20261101,
    "n_patients": 5000,
    "window": {"start_year": 2021, "end_year": 2026},
    "visits": {"max": 8},
    "visit_gap_months": {"Low": 12, "Moderate": 4, "High": 1.5},
    "family_multiplier": 2.0,
    "relatives": {"confirmed_share": 0.70, "unknown_onset_share": 0.20},
    "paths": {"healthy": 0.50, "progressor": 0.25, "already_ill": 0.15, "improver": 0.10},
    "fhrs": {
        "R": {1: 0.50, 2: 0.25, 3: 0.125},
        "S": {"confirmed": 1.0, "not_sure": 0.5},
        "cluster_bonus": 1.0,
        "penalties": {
            "CVD": {"secondhand_smoke": 0.15, "high_salt_diet": 0.15},
            "T2D": {"processed_diet": 0.15, "low_home_activity": 0.15},
            "CRD": {"secondhand_smoke": 0.15, "indoor_mold": 0.15, "coal_heating": 0.15},
            "NEURO": {},
        },
    },
    "drift_per_year": {"hba1c": {"healthy": 0.02, "progressor": 0.20, "improver": -0.10}},
    "missing": {"optional_lab": 0.70, "hospital_only": 0.93},
    "label_window_years": 5,
    "gbd_file": "data/gbd_2023_ph_rates.csv",
    "deaths_file": "data/psa_deaths_2024.csv",
    "pop_file": "data/psa_population_projection.csv",
    "death_year": "2024",
    "lifetime_target": {
        "Major depressive disorder": {"Female": 0.148, "Male": 0.074},
        "Anxiety disorders": {"Female": 0.016, "Male": 0.016},
        "Asthma": {"Female": 0.10, "Male": 0.10},
    },
    "checks": {
        "hypertension": (0.17, 0.23),
        "current_smoker": 0.204,
        "current_depression": 0.033,
    },
}
RNG = np.random.default_rng(CFG["seed"])
print("Setup OK")

Part 0: Prepare disease rates

In [ ]:
# Cell 2 - CODE
AGE_GROUPS = [f"{a}-{a + 4} years" for a in range(15, 95, 5)] + ["95+ years"]

def load_rates(path):
    g = pd.read_csv(path)
    g = g[g["age_name"].isin(AGE_GROUPS) & (g["metric_name"] == "Rate")]
    wide = g.pivot_table(index=["cause_name", "sex_name", "age_name"], 
                         columns="measure_name", values="val").reset_index()
    wide["age_lo"] = wide["age_name"].str.extract(r"^(\d+)").astype(int)
    return pd.DataFrame({
        "disease": wide["cause_name"], "sex": wide["sex_name"], "age_lo": wide["age_lo"],
        "incidence": wide["Incidence"] / 100_000,
        "prevalence": wide["Prevalence"] / 100_000,
    })

def yearly_rate(rates, disease, age, sex, factor=1.0):
    if age < 15: return 0.0
    lo = min(95, age - age % 5)
    row = rates[(rates["disease"] == disease) & (rates["sex"] == sex) & (rates["age_lo"] == lo)]
    return float(row["incidence"].iloc[0]) * factor

def lifetime_risk(rates, disease, sex, factor=1.0, start=15, end=80):
    free = 1.0
    for age in range(start, end):
        free *= 1 - min(yearly_rate(rates, disease, age, sex, factor), 0.99)
    return 1 - free

def calibrate_factor(rates, disease, sex, target, start=15, end=80):
    lo, hi = 0.0, 1.0
    for _ in range(60):
        mid = (lo + hi) / 2
        if lifetime_risk(rates, disease, sex, mid, start, end) < target:
            lo = mid
        else:
            hi = mid
    return (lo + hi) / 2

In [ ]:
%%ipytest -q
# Cell 1 - TEST
import pytest

RAW = pd.read_csv(CFG["gbd_file"])
R = load_rates(CFG["gbd_file"])

def test_keeps_only_the_17_five_year_age_groups():
    assert sorted(R["age_lo"].unique()) == list(range(15, 95, 5)) + [95]

def test_has_7_diseases_2_sexes():
    assert R["disease"].nunique() == 7 and set(R["sex"]) == {"Male", "Female"}
    assert len(R) == 7 * 17 * 2

def test_rate_is_chance_per_person_per_year():
    raw = RAW.query("measure_name == 'Incidence' and cause_name == 'Diabetes mellitus type 2' "
                    "and sex_name == 'Male' and age_name == '50-54 years'")["val"].item()
    assert yearly_rate(R, "Diabetes mellitus type 2", 52, "Male") == pytest.approx(raw / 100_000)

def test_rate_is_zero_under_15():
    assert yearly_rate(R, "Stroke", 10, "Female") == 0.0

def test_age_100_uses_95_plus_group():
    raw = RAW.query("measure_name == 'Incidence' and cause_name == 'Stroke' "
                    "and sex_name == 'Female' and age_name == '95+ years'")["val"].item()
    assert yearly_rate(R, "Stroke", 100, "Female") == pytest.approx(raw / 100_000)

def test_factor_scales_the_rate():
    base = yearly_rate(R, "Asthma", 40, "Male")
    assert yearly_rate(R, "Asthma", 40, "Male", factor=0.25) == pytest.approx(base * 0.25)

def test_raw_depression_rates_give_unrealistic_lifetime_risk():
    assert lifetime_risk(R, "Major depressive disorder", "Female") > 0.80

def test_calibration_hits_the_target_lifetime_risk():
    k = calibrate_factor(R, "Major depressive disorder", "Female", target=0.15)
    assert 0 < k < 1
    assert lifetime_risk(R, "Major depressive disorder", "Female", factor=k) == pytest.approx(0.15, abs=0.001)

In [ ]:
# Cell 3 - RUN
APP_NAME = {"Major depressive disorder": "depression", "Anxiety disorders": "GAD",
            "Asthma": "asthma", "Chronic obstructive pulmonary disease": "COPD",
            "Diabetes mellitus type 2": "T2D"}

RATES = load_rates(CFG["gbd_file"])
RATES["disease"] = RATES["disease"].replace(APP_NAME)

# CVD = ischemic heart disease OR stroke (chance of at least one)
ihd = RATES[RATES["disease"] == "Ischemic heart disease"].set_index(["sex", "age_lo"])
stk = RATES[RATES["disease"] == "Stroke"].set_index(["sex", "age_lo"])
cvd = pd.DataFrame({"incidence": 1 - (1 - ihd["incidence"]) * (1 - stk["incidence"]),
                    "prevalence": ihd["prevalence"] + stk["prevalence"]}).reset_index()
cvd["disease"] = "CVD"
RATES = pd.concat([RATES, cvd[RATES.columns]], ignore_index=True)

TARGET = {APP_NAME.get(d, d): t for d, t in CFG["lifetime_target"].items()}
FACTOR = {(d, s): calibrate_factor(RATES, d, s, t)
          for d, by_sex in TARGET.items() for s, t in by_sex.items()}

# fast lookup: a dict is ~1000x faster than filtering the DataFrame each call
LOOKUP = {(d, s, a): i for d, s, a, i in RATES[["disease", "sex", "age_lo", "incidence"]].itertuples(index=False)}

SEX = {"M": "Male", "F": "Female", "Male": "Male", "Female": "Female"}

def rate_fn(disease, age, sex):
    sex = SEX[sex]
    if age < 15:
        return 0.0
    return LOOKUP[(disease, sex, min(95, age - age % 5))] * FACTOR.get((disease, sex), 1.0)

print("Factors:", FACTOR)
print(sorted(RATES["disease"].unique()))

In [ ]:
%%ipytest -q
# Cell 4 - TEST for the RUN cell (app disease names, CVD, sex codes)
import pytest

def test_rate_fn_knows_every_app_disease_name():
    for d in ["CVD", "T2D", "asthma", "COPD", "depression", "GAD"]:
        for s in ("Male", "Female"):
            assert rate_fn(d, 50, s) > 0

def test_cvd_rate_combines_ischemic_heart_disease_and_stroke():
    ihd = yearly_rate(RATES, "Ischemic heart disease", 60, "Male")
    stroke = yearly_rate(RATES, "Stroke", 60, "Male")
    assert rate_fn("CVD", 60, "Male") == pytest.approx(1 - (1 - ihd) * (1 - stroke))

def test_asthma_calibration_factor_is_used():
    raw = yearly_rate(RATES, "asthma", 40, "Female")
    assert rate_fn("asthma", 40, "Female") == pytest.approx(raw * FACTOR[("asthma", "Female")])

def test_rate_fn_is_zero_under_15():
    assert rate_fn("T2D", 10, "Male") == 0.0

def test_rate_fn_accepts_short_sex_codes_used_by_fhrs():
    assert rate_fn("T2D", 50, "F") == rate_fn("T2D", 50, "Female")

Part 0b: Prepare death rates

In [ ]:
# Cell 2 - CODE
def load_death_rates(deaths_path, pop_path, year="2024"):
    d = pd.read_csv(deaths_path, skiprows=1)
    d["Age Group"] = d["Age Group"].replace({"Under 1": "0-4", "1-4": "0-4", "85 and over": "85+"})
    d = d[~d["Age Group"].isin(["Total", "Not Stated"])]
    d = d.groupby("Age Group", sort=False)[["Male", "Female"]].sum()
    
    p = pd.read_csv(pop_path, skiprows=1)
    rows = []
    for sex in ("Male", "Female"):
        pop = p[p["Sex"] == sex].set_index("Age Group")[year] * 1000
        for ag in d.index:
            lo = 85 if ag == "85+" else int(ag.split("-")[0])
            rows.append({"sex": sex, "age_lo": lo, "rate": d.loc[ag, sex] / pop[ag]})
    return pd.DataFrame(rows)

_DR_LOOKUP = {}

def death_rate(dr, age, sex):
    sex = {"M": "Male", "F": "Female"}.get(sex, sex)
    lo = min(85, age - age % 5)
    if id(dr) not in _DR_LOOKUP:          # fast lookup, built once per table
        _DR_LOOKUP[id(dr)] = {(s, a): r for s, a, r in dr[["sex", "age_lo", "rate"]].itertuples(index=False)}
    return float(_DR_LOOKUP[id(dr)][(sex, lo)])

def life_expectancy(dr, sex):
    m = dr[dr["sex"] == sex].sort_values("age_lo")["rate"].tolist()
    alive, years = 1.0, 0.0
    for mx in m[:-1]:
        q = 5 * mx / (1 + 2.5 * mx)
        years += 5 * alive * (1 - q / 2)
        alive *= 1 - q
    return years + alive / m[-1]

def draw_death_year(birth_year, sex, rng, dr, end_year=2026):
    for year in range(birth_year, end_year + 1):
        if rng.random() < death_rate(dr, year - birth_year, sex):
            return year
    return None   # still alive

In [ ]:
%%ipytest -q
# Cell 1 - TEST
import pytest

DR = load_death_rates(CFG["deaths_file"], CFG["pop_file"], CFG["death_year"])

def test_population_in_thousands_is_converted():
    assert death_rate(DR, 62, "Male") == pytest.approx(42489 / 1_807_000)

def test_under_1_and_1_to_4_are_merged_into_0_to_4():
    assert death_rate(DR, 2, "Male") == pytest.approx((11714 + 3320) / 4_606_000)

def test_age_90_uses_85_plus_group():
    assert death_rate(DR, 90, "Female") == pytest.approx(53786 / 284_000)

def test_rates_rise_with_age_after_30():
    rates = [death_rate(DR, a, "Female") for a in range(30, 90, 5)]
    assert rates == sorted(rates)

def test_life_expectancy_is_believable():
    assert 65 < life_expectancy(DR, "Male") < 75
    assert 70 < life_expectancy(DR, "Female") < 82
    assert life_expectancy(DR, "Female") > life_expectancy(DR, "Male")

def test_death_year_is_after_birth_and_old_people_are_dead():
    rng = np.random.default_rng(3)
    years = [draw_death_year(1900, "Female", rng, DR) for _ in range(200)]
    assert all(y is not None and 1900 <= y <= 2026 for y in years)

def test_young_person_is_usually_alive():
    rng = np.random.default_rng(3)
    alive = [draw_death_year(2000, "Female", rng, DR) is None for _ in range(500)]
    assert np.mean(alive) > 0.9

def test_death_draw_accepts_short_sex_codes():
    assert draw_death_year(1900, "M", np.random.default_rng(1), DR) is not None

In [ ]:
# Cell 3 - RUN
DR = load_death_rates(CFG["deaths_file"], CFG["pop_file"], CFG["death_year"])
print("Male Life Expectancy:", life_expectancy(DR, "Male"))
print("Female Life Expectancy:", life_expectancy(DR, "Female"))
print(DR.head())

Part 0c: Family size

In [ ]:
# Cell 2 - CODE
TFR = {1993: 4.1, 1998: 3.7, 2003: 3.5, 2008: 3.3, 2013: 3.0, 2017: 2.7, 2022: 1.9}

def mean_siblings(birth_year):
    tfr = np.interp(birth_year, list(TFR), list(TFR.values()))
    return float(tfr) - 1

In [ ]:
%%ipytest -q
# Cell 1 - TEST
import pytest

def test_mean_siblings_uses_ndhs_tfr_minus_one():
    assert mean_siblings(1993) == pytest.approx(3.1)

def test_mean_siblings_interpolates_between_surveys():
    assert mean_siblings(2010) == pytest.approx(3.3 + (3.0 - 3.3) * 2 / 5 - 1)

def test_births_before_1993_use_the_1993_value():
    assert mean_siblings(1965) == pytest.approx(3.1)

def test_sourced_targets_can_be_reached():
    for disease, by_sex in TARGET.items():
        for sex, target in by_sex.items():
            k = calibrate_factor(RATES, disease, sex, target)
            assert lifetime_risk(RATES, disease, sex, factor=k) == pytest.approx(target, abs=0.001)

In [ ]:
# Cell 3 - RUN
print("Mean siblings born in 1960:", mean_siblings(1960))
print("Mean siblings born in 2022:", mean_siblings(2022))

Part A: FHRS

In [ ]:
# Cell 2 - CODE
DOMAIN = {"CVD": ["CVD"], "T2D": ["T2D"], "CRD": ["asthma", "COPD"], "NEURO": ["depression", "GAD"]}

def onset_multiplier(disease, age, sex):
    if age is None: return 1.0
    if disease == "CVD":
        if age < 40: return 3.0
        if sex == "M": return 2.0 if age < 55 else 1.0 if age < 65 else 0.75
        return 2.0 if age < 65 else 1.0 if age < 75 else 0.75
    if disease in ("T2D", "COPD"):
        return 3.0 if age < 40 else 2.0 if age < 50 else 1.0 if age < 65 else 0.75
    if disease == "asthma":
        return 3.0 if age < 12 else 2.0 if age <= 25 else 1.0 if age <= 40 else 0.75
    if disease == "depression":
        return 3.0 if age < 18 else 2.0 if age <= 24 else 1.0 if age <= 40 else 0.75
    return 1.0

def fhrs(relatives, exposures, domain, on_date, cfg):
    R, S = cfg["fhrs"]["R"], cfg["fhrs"]["S"]
    bio, confirmed_first = 0.0, {}
    
    for r in relatives:
        for c in r["conditions"]:
            if c["disease"] in DOMAIN[domain] and c["year"] <= on_date.year:
                s = S["confirmed"] if c["confirmed"] else S["not_sure"]
                bio += R[r["degree"]] * onset_multiplier(c["disease"], c["age"], r["sex"]) * s
                if r["degree"] == 1 and c["confirmed"]:
                    confirmed_first[c["disease"]] = confirmed_first.get(c["disease"], 0) + 1
                break
                
    if any(n >= 2 for n in confirmed_first.values()):
        bio += cfg["fhrs"]["cluster_bonus"]
        
    E = 1.0 + sum(v for k, v in cfg["fhrs"]["penalties"][domain].items() if exposures.get(k))
    return round(bio * E, 2)

def band(score):
    if score == 0: return 0
    if score <= 1.75: return 1
    if score <= 3.50: return 2
    if score <= 6.00: return 3
    return 4

In [ ]:
%%ipytest -q
# Cell 1 - TEST
from datetime import date

def rel(degree, sex, *conds):
    return {"degree": degree, "sex": sex, 
            "conditions": [{"disease": d, "year": y, "age": a, "confirmed": c} for d, y, a, c in conds]}

def test_cvd_onset_male_48_is_early():
    assert onset_multiplier("CVD", 48, "M") == 2.0

def test_cvd_onset_female_60_is_still_early():
    assert onset_multiplier("CVD", 60, "F") == 2.0

def test_unknown_onset_counts_as_typical():
    assert onset_multiplier("T2D", None, "F") == 1.0

def test_john_example_scores_4_17():
    relatives = [rel(1, "M", ("CVD", 1998, 48, True)), rel(1, "M", ("CVD", 2016, 38, True)), 
                 rel(2, "M", ("CVD", 2008, 60, False))]
    assert fhrs(relatives, {"secondhand_smoke": True}, "CVD", date(2026, 1, 1), CFG) == 4.17

def test_relative_diagnosed_after_visit_is_not_counted():
    assert fhrs([rel(1, "F", ("T2D", 2024, 64, True))], {}, "T2D", date(2023, 6, 1), CFG) == 0.0

def test_relative_diagnosed_before_visit_is_counted():
    assert fhrs([rel(1, "F", ("T2D", 2024, 64, True))], {}, "T2D", date(2025, 6, 1), CFG) == 0.5

def test_cluster_bonus_needs_two_confirmed_first_degree():
    r = [rel(1, "F", ("T2D", 2000, 55, True)), rel(1, "M", ("T2D", 2001, 55, False))]
    assert fhrs(r, {}, "T2D", date(2026, 1, 1), CFG) == 0.75

def test_crd_counts_asthma_and_copd_without_cluster_bonus():
    r = [rel(1, "F", ("asthma", 1990, 30, True)), rel(1, "M", ("COPD", 2010, 70, True))]
    assert fhrs(r, {}, "CRD", date(2026, 1, 1), CFG) == 0.88

def test_cluster_bonus_when_two_confirmed_have_same_disease():
    r = [rel(1, "F", ("asthma", 1990, 30, True)), rel(1, "M", ("asthma", 1995, 35, True))]
    assert fhrs(r, {}, "CRD", date(2026, 1, 1), CFG) == 2.0

def test_neuro_ignores_household_exposures():
    r = [rel(1, "F", ("depression", 2000, 30, True))]
    assert fhrs(r, {"secondhand_smoke": True}, "NEURO", date(2026, 1, 1), CFG) == 0.5

def test_band_edges_match_guide():
    assert [band(s) for s in [0, 0.05, 1.75, 1.76, 3.50, 3.51, 6.00, 6.01]] == [0, 1, 1, 2, 2, 3, 3, 4]

In [ ]:
# Cell 3 - RUN
johns_relatives = [
    {"degree": 1, "sex": "M", "conditions": [{"disease": "CVD", "year": 1998, "age": 48, "confirmed": True}]},
    {"degree": 1, "sex": "M", "conditions": [{"disease": "CVD", "year": 2016, "age": 38, "confirmed": True}]},
    {"degree": 2, "sex": "M", "conditions": [{"disease": "CVD", "year": 2008, "age": 60, "confirmed": False}]}
]
score = fhrs(johns_relatives, {"secondhand_smoke": True}, "CVD", date(2026, 1, 1), CFG)
print(f"John's FHRS Score: {score}")
print(f"John's Band: {band(score)}")

Part B: Relatives' diseases

In [ ]:
# Cell 2 - CODE
def simulate_family_diseases(relatives, cfg, rng, rate_fn, diseases, end_year=2026):
    first = min(r["birth_year"] for r in relatives)
    for year in range(first, end_year + 1):
        for r in relatives:
            age = year - r["birth_year"]
            if age < 0 or (r.get("death_year") and year > r["death_year"]):
                continue
            for d in diseases:
                if any(c["disease"] == d for c in r["conditions"]):
                    continue
                rate = rate_fn(d, age, r["sex"])
                
                if any(o is not r and o["degree"] == 1 and 
                       any(c["disease"] == d and c["year"] < year for c in o["conditions"]) 
                       for o in relatives):
                    rate *= cfg["family_multiplier"]
                    
                if rng.random() < min(rate, 0.99):
                    r["conditions"].append({"disease": d, "year": year, "age": age, "confirmed": True})

In [ ]:
%%ipytest -q
# Cell 1 - TEST

def test_disease_clusters_in_families():
    rng = np.random.default_rng(7)
    flat = lambda disease, age, sex: 0.01 if age >= 30 else 0.0
    fams = []
    
    for _ in range(1500):
        fam = [{"degree": 1, "sex": "F", "birth_year": 1950, "conditions": []},
               {"degree": 1, "sex": "M", "birth_year": 1950, "conditions": []},
               {"degree": 1, "sex": "F", "birth_year": 1975, "conditions": []}]
        simulate_family_diseases(fam, CFG, rng, flat, diseases=["T2D"])
        fams.append(fam)
        
    has = lambda r: any(c["disease"] == "T2D" for c in r["conditions"])
    with_parent = [f for f in fams if has(f[0]) or has(f[1])]
    without = [f for f in fams if not (has(f[0]) or has(f[1]))]
    
    assert np.mean([has(f[2]) for f in with_parent]) > 1.3 * np.mean([has(f[2]) for f in without])

def test_dead_relative_gets_no_new_disease():
    fam = [{"degree": 1, "sex": "M", "birth_year": 1900, "death_year": 1950, "conditions": []}]
    simulate_family_diseases(fam, CFG, np.random.default_rng(1), lambda d, a, s: 0.5 if a >= 60 else 0.0, ["T2D"])
    assert all(c["year"] <= 1950 for c in fam[0]["conditions"])

In [ ]:
# Cell 3 - RUN
rng = np.random.default_rng(7)
test_fam = [{"degree": 1, "sex": "F", "birth_year": 1960, "conditions": []},
            {"degree": 1, "sex": "M", "birth_year": 1962, "conditions": []}]
simulate_family_diseases(test_fam, CFG, rng, rate_fn, diseases=["T2D", "CVD"])
print("Simulated Family Conditions:", test_fam)

Part C: Visit schedule

In [ ]:
# Cell 2 - CODE
def add_months(d, months):
    m = d.month - 1 + int(round(months))
    return date(d.year + m // 12, m % 12 + 1, 1)

def schedule_visits(rng, cfg, tier_fn=None):
    tier_fn = tier_fn or (lambda d: "Low")
    d = date(cfg["window"]["start_year"] + int(rng.integers(0, 2)), int(rng.integers(1, 13)), 1)
    end = date(cfg["window"]["end_year"], 12, 31)
    dates = []
    
    while d <= end and len(dates) < cfg["visits"]["max"]:
        dates.append(d)
        gap = max(1, cfg["visit_gap_months"][tier_fn(d)] * rng.uniform(0.8, 1.2))
        d = add_months(d, gap)
    return dates

In [ ]:
%%ipytest -q
# Cell 1 - TEST
from datetime import date

def test_visit_dates_go_forward_inside_window():
    dates = schedule_visits(np.random.default_rng(1), CFG)
    assert dates == sorted(dates)
    assert all(date(2021, 1, 1) <= d <= date(2026, 12, 31) for d in dates)
    assert 1 <= len(dates) <= 8

In [ ]:
# Cell 3 - RUN
rng = np.random.default_rng(1)
visits = schedule_visits(rng, CFG)
print("Patient Visit Schedule:", visits)

Part D: Outcome labels

In [ ]:
# Cell 2 - CODE
def label_for(diagnosis_date, index_date, years):
    if diagnosis_date is None:
        return 0
    if diagnosis_date <= index_date:
        return None
    end = date(index_date.year + years, index_date.month, index_date.day)
    return int(diagnosis_date <= end)

In [ ]:
%%ipytest -q
# Cell 1 - TEST
from datetime import date

def test_label_is_1_when_diagnosed_inside_window():
    assert label_for(date(2024, 5, 1), date(2021, 3, 1), 5) == 1

def test_label_is_0_when_diagnosed_after_window():
    assert label_for(date(2027, 5, 1), date(2021, 3, 1), 5) == 0

def test_already_ill_at_index_is_excluded():
    assert label_for(date(2020, 1, 1), date(2021, 3, 1), 5) is None

In [ ]:
# Cell 3 - RUN
index = date(2021, 1, 1)
dx = date(2025, 6, 1)
print(f"Diagnosed {dx}, Index {index} -> Label:", label_for(dx, index, 5))

Part E: Patients, values, and saving

Values come from the CDSS Data Dictionary (oct31): min/max from the Parameter Ingestion Tables, bands from the Disease Risk Threshold tables. Shares marked AD are author-defined.

In [ ]:
# Cell 2 - CODE
from pathlib import Path
# ---------------------------------------------------------------------------
# E1. Value table from the CDSS Data Dictionary (oct31)
# min/max = Parameter Ingestion Tables. low/mod/high = draw ranges INSIDE the
# Low / Moderate / High risk bands of the Disease Risk Threshold tables
# (narrowed to typical values: author-defined). kind: req = Required,
# opt = Optional lab, phys = "To confirm with physician" (hospital test).
# drv = which disease pushes the value up: {disease: band when diagnosed}.
# ---------------------------------------------------------------------------
CVD_ = {"CVD": "high"}
HTN_ = {"CVD": "high", "HTN": "high"}
T2D_ = {"T2D": "high"}
CRD_ = {"asthma": "high", "COPD": "high"}
COPD_ = {"COPD": "high", "asthma": "mod"}
AST_ = {"asthma": "high"}
DEP_ = {"depression": "high"}
GAD_ = {"GAD": "high"}
NEU_ = {"depression": "high", "GAD": "high"}
NONE_ = {}

def v(mn, mx, low, mod, high, drv, kind, dec=1):
    return {"min": mn, "max": mx, "low": low, "mod": mod, "high": high, "drv": drv, "kind": kind, "dec": dec}

SPEC = {
    # 1. CVD
    "sbp":               v(70, 220, (100, 119), (120, 139), (140, 180), HTN_, "req", 0),
    "dbp":               v(40, 130, (65, 79), (80, 89), (90, 110), HTN_, "req", 0),
    "heart_rate":        v(40, 180, (60, 80), (81, 99), (100, 120), CVD_, "req", 0),
    "total_cholesterol": v(100, 400, (140, 199), (200, 239), (240, 300), CVD_, "opt", 0),
    "hdl":               v(15, 120, (60, 80), (40, 59), (30, 39), CVD_, "opt", 0),
    "ldl":               v(30, 300, (70, 99), (100, 159), (160, 220), CVD_, "opt", 0),
    "triglycerides":     v(40, 1000, (70, 149), (150, 199), (200, 400), T2D_, "opt", 0),
    "hs_crp":            v(0.1, 50, (0.2, 0.9), (1.0, 3.0), (3.1, 10), CVD_, "opt", 1),
    "creatinine":        v(0.4, 10, (0.6, 1.2), (1.3, 1.9), (2.0, 4.0), CVD_, "opt", 2),
    "egfr":              v(5, 140, (90, 120), (60, 89), (30, 59), CVD_, "opt", 0),
    "lvef":              v(15, 75, (55, 70), (41, 52), (25, 40), CVD_, "phys", 0),
    "ctni":              v(0.01, 50, (0.01, 0.03), (0.04, 0.39), (0.40, 2.0), CVD_, "phys", 2),
    "nt_probnp":         v(10, 35000, (20, 124), (125, 449), (450, 3000), CVD_, "phys", 0),
    "sodium_mg":         v(500, 10000, (1500, 2300), (2301, 4000), (4001, 6000), HTN_, "req", 0),
    "activity_min_week": v(0, 1050, (150, 400), (60, 149), (0, 59), CVD_, "req", 0),
    # 2. T2D
    "fpg":               v(50, 450, (75, 99), (100, 125), (126, 250), T2D_, "opt", 0),
    "pg_2h":             v(60, 600, (90, 139), (140, 199), (200, 350), T2D_, "opt", 0),
    "hba1c":             v(4.0, 15.0, (4.6, 5.6), (5.7, 6.4), (6.5, 10.0), T2D_, "opt", 1),
    "insulin":           v(1.0, 100.0, (3.0, 12.0), (12.1, 24.9), (25.0, 50.0), T2D_, "opt", 1),
    "c_peptide":         v(0.1, 12.0, (1.1, 3.0), (3.1, 4.4), (4.5, 8.0), T2D_, "opt", 1),
    "waist_cm":          v(50, 160, (65, 79), (80, 101), (102, 125), T2D_, "req", 0),
    "whr":               v(0.60, 1.40, (0.75, 0.84), (0.85, 0.89), (0.90, 1.05), T2D_, "req", 2),
    "uacr":              v(1, 3500, (5, 29), (30, 299), (300, 1000), T2D_, "opt", 0),
    "uric_acid":         v(1.5, 14.0, (3.5, 5.9), (6.0, 6.9), (7.0, 10.0), T2D_, "opt", 1),
    "carbs_g":           v(50, 600, (130, 250), (251, 350), (351, 500), T2D_, "req", 0),
    "sedentary_hours":   v(1.0, 18.0, (1.0, 3.9), (4.0, 7.9), (8.0, 12.0), T2D_, "req", 1),
    # 3. CRD
    "fev1":              v(0.5, 5.5, (3.0, 4.5), (1.8, 2.9), (0.8, 1.7), CRD_, "opt", 2),
    "fvc":               v(1.0, 7.0, (3.5, 5.5), (2.2, 3.4), (1.2, 2.1), CRD_, "opt", 2),
    "fev1_fvc":          v(25.0, 95.0, (72.0, 85.0), (60.0, 69.9), (40.0, 59.9), COPD_, "opt", 1),
    "fev1_pct_pred":     v(15.0, 120.0, (80.0, 105.0), (50.0, 79.9), (30.0, 49.9), CRD_, "opt", 1),
    "pef":               v(50, 800, (400, 600), (250, 399), (120, 249), CRD_, "opt", 0),
    "eosinophils":       v(0, 2500, (50, 149), (150, 299), (300, 800), AST_, "opt", 0),
    "feno":              v(5, 150, (8, 24), (25, 49), (50, 100), AST_, "opt", 0),
    "spo2":              v(65.0, 100.0, (95.0, 100.0), (90.0, 94.9), (85.0, 89.9), CRD_, "opt", 1),
    "pao2":              v(40, 110, (80, 100), (60, 79), (50, 59), CRD_, "phys", 0),
    "paco2":             v(25, 90, (35, 45), (46, 50), (51, 60), COPD_, "phys", 0),
    "mmrc":              v(0, 4, (0, 1), (2, 2), (3, 4), CRD_, "req", 0),
    "act":               v(5, 25, (20, 25), (16, 19), (8, 15), AST_, "phys", 0),
    "cat":               v(0, 40, (0, 9), (10, 20), (21, 30), COPD_, "phys", 0),
    "ige":               v(2, 3000, (10, 99), (100, 299), (300, 1000), AST_, "opt", 0),
    "exacerbations_year": v(0, 15, (0, 0), (1, 1), (2, 4), CRD_, "phys", 0),
    # 4. NEURO
    "phq9":              v(0, 27, (0, 4), (5, 9), (10, 20), DEP_, "req", 0),
    "gad7":              v(0, 21, (0, 4), (5, 9), (10, 18), GAD_, "req", 0),
    "psqi":              v(0, 21, (2, 5), (6, 10), (11, 16), NEU_, "opt", 0),
    "sleep_hours":       v(1.0, 16.0, (7.0, 9.0), (5.0, 6.9), (3.0, 4.9), NEU_, "req", 1),
    "tsh":               v(0.01, 100.0, (0.5, 4.25), (4.26, 9.99), (10.0, 20.0), NONE_, "opt", 2),
    "vitamin_b12":       v(50, 2000, (250, 900), (100, 199), (60, 99), NONE_, "opt", 0),
}

# Author-defined shares (AD). Change here, then re-run Notebook 2.
AD = {
    "baseline_mod_share": 0.20,      # healthy person, one value already borderline
    "former_smoker": 0.10,           # current smoker share comes from CFG["checks"] (FNRI 20.4%)
    "gdm_if_t2d": 0.10, "gdm_other": 0.03,
    "acanthosis_if_t2d": 0.40, "acanthosis_other": 0.05,
    "allergic_rhinitis_if_asthma": 0.60, "allergic_rhinitis_other": 0.15,
    "occupational_dust": 0.15, "alcohol_use": 0.50, "tbi": 0.03,
    "exposure": {"secondhand_smoke": 0.35, "high_salt_diet": 0.20, "processed_diet": 0.20,
                 "low_home_activity": 0.20, "indoor_mold": 0.15, "coal_heating": 0.05},
    "neuro_episode_years": 2,        # PHQ-9/GAD-7 stay high this long after diagnosis
    "height_cm": {"M": (163.5, 6.0), "F": (151.5, 5.5)},
}
ALL_DISEASES = ["CVD", "T2D", "asthma", "COPD", "depression", "GAD"]
APP_DOMAINS = {"CVD": ["CVD"], "T2D": ["T2D"], "CRD": ["asthma", "COPD"], "NEURO": ["depression", "GAD"]}

def dx_date(year):
    return None if year is None else date(int(year), 7, 1)   # diagnosis = middle of that year

# ---------------------------------------------------------------------------
# E2. Ages and sexes from PSA 2024 projected population (adults 20-79)
# ---------------------------------------------------------------------------
def load_age_weights(pop_path, year="2024"):
    p = pd.read_csv(pop_path, skiprows=1)
    p = p[p["Sex"].isin(["Male", "Female"]) & p["Age Group"].str.match(r"^[2-7]\d-")]
    p = p.assign(lo=p["Age Group"].str.split("-").str[0].astype(int), w=p[year])
    return p[["Sex", "lo", "w"]].reset_index(drop=True)

AGE_W = load_age_weights(CFG["pop_file"], CFG["death_year"])

def hypertension_chance(age):
    # rises with age; tuned so adults average about 20% (FNRI target in CFG["checks"]) - shape AD
    return float(np.clip(0.02 + 0.007 * (age - 20), 0.02, 0.45))

# ---------------------------------------------------------------------------
# E3. make_patient: one patient + family tree + diseases over time
# ---------------------------------------------------------------------------
def _relative(rel, degree, sex, birth_year, rng):
    return {"relation": rel, "degree": degree, "sex": sex, "birth_year": int(birth_year),
            "death_year": draw_death_year(int(birth_year), sex, rng, DR), "conditions": []}

def make_family(birth_year, rng):
    fam = []
    for rel, sex in (("father", "M"), ("mother", "F")):
        by = birth_year - int(rng.integers(20, 41))
        fam.append(_relative(rel, 1, sex, by, rng))
        for gsex in ("M", "F"):
            fam.append(_relative(f"{rel}'s {'father' if gsex == 'M' else 'mother'}", 2, gsex,
                                 by - int(rng.integers(20, 41)), rng))
    for _ in range(min(8, rng.poisson(mean_siblings(birth_year)))):
        fam.append(_relative("sibling", 1, rng.choice(["M", "F"]),
                             birth_year + int(rng.integers(-10, 11)), rng))
    return fam

def make_patient(pid, rng, cfg=CFG):
    row = AGE_W.iloc[rng.choice(len(AGE_W), p=AGE_W["w"] / AGE_W["w"].sum())]
    sex = "M" if row["Sex"] == "Male" else "F"
    age0 = int(row["lo"]) + int(rng.integers(0, 5))           # age on 1 Jan of the window start
    birth_year = cfg["window"]["start_year"] - age0

    me = {"relation": "self", "degree": 0, "sex": sex, "birth_year": birth_year,
          "death_year": None, "conditions": []}
    family = make_family(birth_year, rng)
    simulate_family_diseases(family + [me], cfg, rng, rate_fn, ALL_DISEASES,
                             end_year=cfg["window"]["end_year"] + 1)
    for r in family:                                           # what the patient knows
        for c in r["conditions"]:
            c["confirmed"] = bool(rng.random() < cfg["relatives"]["confirmed_share"])
            if rng.random() < cfg["relatives"]["unknown_onset_share"]:
                c["age"] = None

    dx = {c["disease"]: c["year"] for c in me["conditions"]}
    dx = {d: dx.get(d) for d in ALL_DISEASES}
    dx["HTN"] = 1900 if rng.random() < hypertension_chance(age0 + 3) else None

    smoke_p = cfg["checks"]["current_smoker"]
    s = rng.random()
    status = "current" if s < smoke_p else "former" if s < smoke_p + AD["former_smoker"] else "never"
    ppd = float(rng.uniform(0.25, 1.5))
    years = max(0, age0 - 18) if status == "current" else float(rng.uniform(2, 20)) if status == "former" else 0
    has = lambda d: dx[d] is not None
    female = sex == "F"
    mu, sd = AD["height_cm"][sex]
    return {
        "patient_id": pid, "sex": sex, "birth_year": birth_year, "relatives": family, "dx": dx,
        "height_cm": round(float(np.clip(rng.normal(mu, sd), 135, 195)), 1),
        "u": {k: float(rng.random()) for k in list(SPEC) + ["bmi"]},          # personal position in a band
        "base_mod": {k: bool(rng.random() < AD["baseline_mod_share"]) for k in list(SPEC) + ["bmi"]},
        "smoking_status": status, "packs_per_day": ppd if status != "never" else 0.0,
        "pack_years_start": min(120.0, ppd * years) if status != "never" else 0.0,
        "exposures": {k: bool(rng.random() < p) for k, p in AD["exposure"].items()},
        "gestational_diabetes": int(female and rng.random() < (AD["gdm_if_t2d"] if has("T2D") else AD["gdm_other"])),
        "acanthosis_nigricans": int(rng.random() < (AD["acanthosis_if_t2d"] if has("T2D") else AD["acanthosis_other"])),
        "allergic_rhinitis": int(rng.random() < (AD["allergic_rhinitis_if_asthma"] if has("asthma") else AD["allergic_rhinitis_other"])),
        "childhood_asthma": int(has("asthma") and dx["asthma"] - birth_year < 18),
        "occupational_dust": int(rng.random() < AD["occupational_dust"]),
        "alcohol_use": int(rng.random() < AD["alcohol_use"]),
        "tbi_history": int(rng.random() < AD["tbi"]),
    }

# ---------------------------------------------------------------------------
# E4. make_visit_values: one visit's measurements
# ---------------------------------------------------------------------------
LEVELS = ["low", "mod", "high"]

def level_for(p, name, d):
    """Band of a value on date d: high after diagnosis, mod in the 5 years before, else personal baseline."""
    best = "mod" if p["base_mod"][name] else "low"
    for disease, lvl in SPEC_DRV(name).items():
        when = dx_date(p["dx"].get(disease))
        if when is None:
            continue
        if when <= d:
            if disease in ("depression", "GAD") and d.year - when.year > AD["neuro_episode_years"]:
                lvl = "mod"                                     # episode over, still not fully well
        elif when.year - d.year <= 5:
            lvl = "mod"                                         # not yet diagnosed, already drifting
        else:
            continue
        best = max(best, lvl, key=LEVELS.index)
    return best

def SPEC_DRV(name):
    return T2D_ if name == "bmi" else SPEC[name]["drv"]

def ckd_epi_2021(creatinine, age, sex):
    """eGFR (mL/min/1.73m2), CKD-EPI 2021 race-free equation. sex: 'M'/'F' or 'Male'/'Female'."""
    female = sex in ("F", "Female")
    k, a = (0.7, -0.241) if female else (0.9, -0.302)
    s = creatinine / k
    return 142 * min(s, 1) ** a * max(s, 1) ** -1.2 * 0.9938 ** age * (1.012 if female else 1)

def draw(band, u):
    a, b = band
    return a + (b - a) * u

def make_visit_values(p, d, rng, cfg=CFG):
    age = d.year - p["birth_year"]
    labs_done = rng.random() >= cfg["missing"]["optional_lab"]
    hospital_done = rng.random() >= cfg["missing"]["hospital_only"]
    out = {"age": age}
    for name, s in SPEC.items():
        if (s["kind"] == "opt" and not labs_done) or (s["kind"] == "phys" and not hospital_done):
            out[name] = np.nan
            continue
        u = float(np.clip(p["u"][name] + rng.normal(0, 0.05), 0, 1))
        val = draw(s[level_for(p, name, d)], u)
        val = float(np.clip(round(val, s["dec"]), s["min"], s["max"]))
        out[name] = int(val) if s["dec"] == 0 else val
    # FEV1 from FVC and the drawn ratio, then the ratio from the rounded values (computed field)
    if not np.isnan(out["fvc"]):
        out["fev1"] = float(np.clip(round(out["fvc"] * out["fev1_fvc"] / 100, 2), 0.5, 5.5))
        out["fev1_fvc"] = round(out["fev1"] / out["fvc"] * 100, 1)
    # eGFR from creatinine, age and sex: CKD-EPI 2021 (computed field)
    if not np.isnan(out["creatinine"]):
        out["egfr"] = int(np.clip(round(ckd_epi_2021(out["creatinine"], age, p["sex"])), 5, 140))
    # BMI from weight and height (dictionary: computed)
    bmi_band = {"low": (18.5, 24.9), "mod": (25.0, 29.9), "high": (30.0, 40.0)}[level_for(p, "bmi", d)]
    h = p["height_cm"] / 100
    target = draw(bmi_band, float(np.clip(p["u"]["bmi"] + rng.normal(0, 0.03), 0, 1)))
    out["height_cm"] = p["height_cm"]
    out["weight_kg"] = round(target * h * h, 1)
    out["bmi"] = round(out["weight_kg"] / h ** 2, 1)
    # HOMA-IR from FPG and insulin (dictionary: computed)
    out["homa_ir"] = (round(float(np.clip(out["fpg"] * out["insulin"] / 405, 0.2, 25.0)), 1)
                      if not (np.isnan(out["fpg"]) or np.isnan(out["insulin"])) else np.nan)
    # smoking: pack-years only go up for current smokers
    years_since_start = d.year - cfg["window"]["start_year"]
    extra = p["packs_per_day"] * years_since_start if p["smoking_status"] == "current" else 0.0
    out["pack_years"] = round(min(120.0, p["pack_years_start"] + extra), 1)
    out["smoking_status"] = p["smoking_status"]
    for k in ("gestational_diabetes", "acanthosis_nigricans", "allergic_rhinitis", "childhood_asthma",
              "occupational_dust", "alcohol_use", "tbi_history"):
        out[k] = p[k]
    return out

# ---------------------------------------------------------------------------
# E5. Training rows (index date = first visit) and the generator
# ---------------------------------------------------------------------------
def first_dx(p, domain):
    years = [p["dx"][d] for d in APP_DOMAINS[domain] if p["dx"].get(d) is not None]
    return dx_date(min(years)) if years else None

def training_rows(p, index_date, cfg=CFG):
    rows = []
    for domain in APP_DOMAINS:
        lab = label_for(first_dx(p, domain), index_date, cfg["label_window_years"])
        if lab is not None:                                     # already ill at index -> left out
            rows.append({"patient_id": p["patient_id"], "domain": domain,
                         "index_date": index_date, "label": lab})
    return rows

def tier_for(p, d):
    if any(w is not None and w <= d for w in (dx_date(p["dx"][x]) for x in ALL_DISEASES)):
        return "High"
    if max(band(fhrs(p["relatives"], p["exposures"], dom, d, CFG)) for dom in DOMAIN) >= 3:
        return "Moderate"
    return "Low"

def generate(cfg=CFG, n=None, out_dir="output"):
    n = cfg["n_patients"] if n is None else n
    rows_p, rows_r, rows_v, rows_t = [], [], [], []
    for pid in range(n):
        rng = np.random.default_rng([cfg["seed"], pid])
        p = make_patient(pid, rng, cfg)
        visits = schedule_visits(rng, cfg, tier_fn=lambda d: tier_for(p, d))
        for i, d in enumerate(visits):
            vals = make_visit_values(p, d, rng, cfg)
            for dom in DOMAIN:
                score = fhrs(p["relatives"], p["exposures"], dom, d, cfg)
                vals[f"fhrs_{dom}"], vals[f"fhrs_band_{dom}"] = score, band(score)
                w = first_dx(p, dom)
                vals[f"has_{dom}"] = int(w is not None and w <= d)
            rows_v.append({"patient_id": pid, "visit_no": i + 1, "visit_date": d, **vals})
        rows_t += training_rows(p, visits[0], cfg)
        rows_p.append({"patient_id": pid, "sex": "Male" if p["sex"] == "M" else "Female",
                       "birth_year": p["birth_year"], "height_cm": p["height_cm"],
                       "smoking_status": p["smoking_status"],
                       **{k: p[k] for k in ("gestational_diabetes", "acanthosis_nigricans", "allergic_rhinitis",
                                            "childhood_asthma", "occupational_dust", "alcohol_use", "tbi_history")},
                       **{f"exp_{k}": int(x) for k, x in p["exposures"].items()},
                       **{f"true_dx_year_{k}": y for k, y in p["dx"].items()},
                       "is_synthetic": True})
        for j, r in enumerate(p["relatives"]):
            base = {"patient_id": pid, "relative_no": j + 1, "relation": r["relation"], "degree": r["degree"],
                    "sex": r["sex"], "birth_year": r["birth_year"], "death_year": r["death_year"]}
            conds = r["conditions"] or [{"disease": None, "year": None, "age": None, "confirmed": None}]
            for c in conds:
                rows_r.append({**base, "disease": c["disease"], "dx_year": c["year"],
                               "dx_age": c["age"], "confirmed": c["confirmed"]})
    out = {"patients": pd.DataFrame(rows_p), "relatives": pd.DataFrame(rows_r),
           "visits": pd.DataFrame(rows_v), "training": pd.DataFrame(rows_t)}
    if out_dir is not None:
        Path(out_dir).mkdir(parents=True, exist_ok=True)
        for name, df in out.items():
            df.to_csv(Path(out_dir) / f"{name}.csv", index=False)
    return out

In [ ]:
%%ipytest -q
# Cell 1 - TEST
import copy
import pytest

def small_cfg(**over):
    c = copy.deepcopy(CFG)
    c.update(over)
    return c

@pytest.fixture(scope="module")
def sample():
    return generate(CFG, n=150, out_dir=None)

def test_every_value_stays_inside_its_dictionary_min_max_range(sample):
    v = sample["visits"]
    for name, s in SPEC.items():
        col = v[name].dropna()
        assert col.between(s["min"], s["max"]).all(), name
    assert v["bmi"].between(12, 60).all() and v["homa_ir"].dropna().between(0.2, 25).all()

def test_bmi_equals_weight_divided_by_height_squared_rounded_to_0_1(sample):
    v = sample["visits"]
    assert (v["bmi"] == (v["weight_kg"] / (v["height_cm"] / 100) ** 2).round(1)).all()

def test_a_progressors_hba1c_is_higher_at_the_last_visit_than_at_the_first():
    cfg = small_cfg(missing={"optional_lab": 0.0, "hospital_only": 0.0})
    rng = np.random.default_rng(5)
    p = make_patient(1, rng, cfg)
    p["dx"]["T2D"] = 2024                       # becomes diabetic inside the window
    first = make_visit_values(p, date(2021, 3, 1), rng, cfg)
    last = make_visit_values(p, date(2026, 3, 1), rng, cfg)
    assert last["hba1c"] > first["hba1c"]
    assert last["hba1c"] >= 6.5 > first["hba1c"]

def test_gestational_diabetes_appears_only_for_female_patients(sample):
    pts = sample["patients"]
    assert (pts.loc[pts["sex"] == "Male", "gestational_diabetes"] == 0).all()
    assert pts.loc[pts["sex"] == "Female", "gestational_diabetes"].sum() > 0

def test_pack_years_never_go_down_between_visits(sample):
    v = sample["visits"].sort_values(["patient_id", "visit_date"])
    assert (v.groupby("patient_id")["pack_years"].diff().dropna() >= 0).all()

def test_required_values_are_never_missing(sample):
    v = sample["visits"]
    for name, s in SPEC.items():
        if s["kind"] == "req":
            assert v[name].notna().all(), name

def test_age_is_18_to_100_at_every_visit(sample):
    assert sample["visits"]["age"].between(18, 100).all()

def test_patient_already_ill_at_index_has_no_training_row():
    rng = np.random.default_rng(9)
    p = make_patient(2, rng, CFG)
    p["dx"]["CVD"] = 2000
    rows = training_rows(p, date(2021, 5, 1), CFG)
    assert "CVD" not in [r["domain"] for r in rows]

def test_generator_writes_four_linked_files(tmp_path):
    generate(CFG, n=20, out_dir=tmp_path)
    files = {f: pd.read_csv(tmp_path / f) for f in ["patients.csv", "relatives.csv", "visits.csv", "training.csv"]}
    ids = set(files["patients.csv"]["patient_id"])
    assert len(ids) == 20
    for f in ["relatives.csv", "visits.csv", "training.csv"]:
        assert set(files[f]["patient_id"]) <= ids

def expected_egfr(creatinine, age, sex):     # written separately from the generator on purpose
    k, a = (0.7, -0.241) if sex == "Female" else (0.9, -0.302)
    s = creatinine / k
    return 142 * min(s, 1) ** a * max(s, 1) ** -1.2 * 0.9938 ** age * (1.012 if sex == "Female" else 1)

def test_fev1_fvc_ratio_is_computed_from_fev1_and_fvc(sample):
    v = sample["visits"].dropna(subset=["fev1", "fvc"])
    assert len(v) > 0
    assert ((v["fev1_fvc"] - (v["fev1"] / v["fvc"] * 100).round(1)).abs() <= 0.05).all()

def test_egfr_is_computed_from_creatinine_age_and_sex(sample):
    v = sample["visits"].merge(sample["patients"][["patient_id", "sex"]], on="patient_id").dropna(subset=["creatinine"])
    assert len(v) > 0
    expected = [min(140, max(5, round(expected_egfr(c, a, s)))) for c, a, s in zip(v["creatinine"], v["age"], v["sex"])]
    assert (v["egfr"].values == np.array(expected)).all()

In [ ]:
# Cell 3 - RUN  (5,000 patients: about 1-2 minutes)
OUT = generate(CFG, out_dir="output")
for name, df in OUT.items():
    print(f"output/{name}.csv: {len(df):,} rows, {df.shape[1]} columns")